<a href="https://colab.research.google.com/github/pavishanth-sujeevan/E.motion-/blob/llm/llama3_tamil_therapy_2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Tamil Therapy LLM - FIXED Version (LLaMA 3.2-3B)

##  CELL 1: Install & Check GPU

In [1]:
!pip install -q -U transformers peft accelerate bitsandbytes trl datasets
!pip install -q -U evaluate rouge_score scikit-learn seaborn matplotlib

import torch
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, confusion_matrix

print("="*70)
print("GPU CHECK")
print("="*70)
print(f"PyTorch: {torch.__version__}")
print(f"CUDA: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
print("="*70)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.6/47.6 MB 45.0 MB/s eta 0:00:00
GPU CHECK
PyTorch: 2.10.0+cu128
CUDA: True
GPU: NVIDIA A100-SXM4-40GB
Memory: 42.4 GB


##  CELL 2: Mount Drive & Load Tamil Data

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# Data paths
DATA_DIR = "/content/drive/MyDrive/processed_tamil"
TRAIN_FILE = f"{DATA_DIR}/train_ta.jsonl"
VAL_FILE = f"{DATA_DIR}/val_ta.jsonl"

import os
print("\nChecking files...")
print(f"Train: {'Yes' if os.path.exists(TRAIN_FILE) else 'No'}")
print(f"Val: {'Yes' if os.path.exists(VAL_FILE) else 'No'}")

Mounted at /content/drive

Checking files...
Train: Yes
Val: Yes


## CELL 3: Model Config - LLaMA 3.2-3B (Better than Gemma for Tamil!)

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from trl import SFTTrainer, SFTConfig
from datasets import Dataset
import json

# FIXED MODEL CHOICE
MODEL_ID = "meta-llama/Llama-3.2-3B-Instruct"
OUTPUT_DIR = "./llama3-tamil-therapy-fixed"

print("="*70)
print("MODEL: LLaMA 3.2-3B-Instruct")
print("="*70)

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16,
    bnb_4bit_use_double_quant=True,
)

print("\n Config ready")

MODEL: LLaMA 3.2-3B-Instruct

 Config ready


##  CELL 4: Load LLaMA with Optimized LoRA

In [ ]:
from google.colab import userdata
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

# Fetch the secret key from Colab's "Secrets" (ensure the name matches your key)
HF_TOKEN = userdata.get('HF_TOKEN')

print("="*70)
print("LOADING LLAMA 3.2-3B")
print("="*70)

print("\n1/3: Loading model...")
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto",
    torch_dtype=torch.bfloat16,
    trust_remote_code=True,
    token=HF_TOKEN  # Using your loaded secret key
)
model = prepare_model_for_kbit_training(model)
print("    Model loaded")

print("\n2/3: Loading tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(
    MODEL_ID,
    trust_remote_code=True,
    token=HF_TOKEN  # Using your loaded secret key
)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"
print("    Tokenizer loaded")

# Test Tamil tokenization
test_tamil = "எனக்கு மிகவும் வருத்தமாக உள்ளது"
tokens = tokenizer.encode(test_tamil)
print(f"\n   Tamil test: {test_tamil}")
print(f"   Tokens: {len(tokens)}")
print(f"   Tamil works!")



print("\n3/3: Applying LoRA (optimized for Tamil)...")
peft_config = LoraConfig(
    r=96,              # INCREASED for Tamil linguistic complexity
    lora_alpha=192,    # 2x scaling (common practice)
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)
model = get_peft_model(model, peft_config)
model.print_trainable_parameters()
print("\n✅ MODEL READY")

LOADING LLAMA 3.2-3B

1/3: Loading model...


config.json:   0%|          | 0.00/878 [00:00<?, ?B/s]

`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

...

    Model loaded

2/3: Loading tokenizer...


tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/296 [00:00<?, ?B/s]

    Tokenizer loaded

   Tamil test: எனக்கு மிகவும் வருத்தமாக உள்ளது
   Tokens: 44
   Tamil works!

3/3: Applying LoRA (optimized for Tamil)...
trainable params: 145,883,136 || all params: 3,358,632,960 || trainable%: 4.3435

✅ MODEL READY


## CELL 5: Load Data with ROBUST Preprocessing

In [ ]:
print("="*70)
print("LOADING TAMIL DATA (ROBUST)")
print("="*70)

def extract_user_input_robust(prompt):
    """
    ROBUSTLY extract user input from various prompt formats.
    This prevents the 'user' output bug!
    """
    # Try pattern 1: "User: {text} Assistant:"
    if 'User:' in prompt and 'Assistant:' in prompt:
        parts = prompt.split('User:')
        if len(parts) > 1:
            user_part = parts[1]
            if 'Assistant:' in user_part:
                text = user_part.split('Assistant:')[0].strip()
                if text:  # Make sure it's not empty
                    return text

    # Try pattern 2: Just "User: {text}"
    if 'User:' in prompt:
        parts = prompt.split('User:')
        if len(parts) > 1:
            text = parts[1].strip()
            if text:
                return text

    # Fallback: return entire prompt
    return prompt.strip()

def load_tamil_data(filepath):
    """Load Tamil data with emotion extraction"""
    data = []
    with open(filepath, 'r', encoding='utf-8') as f:
        for i, line in enumerate(f, 1):
            try:
                entry = json.loads(line)

                # Extract emotion
                prompt = entry.get('prompt', '')
                if 'feeling' in prompt.lower():
                    try:
                        emotion = prompt.lower().split('feeling')[1].split('.')[0].strip()
                        entry['emotion'] = emotion
                    except:
                        entry['emotion'] = 'unknown'
                else:
                    entry['emotion'] = 'unknown'

                data.append(entry)
            except:
                print(f"   ⚠️ Skipped line {i}")
    return data

# Load
print("\nLoading training data...")
train_data = load_tamil_data(TRAIN_FILE)
print(f"    {len(train_data)} samples")

print("\nLoading validation data...")
val_data = load_tamil_data(VAL_FILE)
print(f"    {len(val_data)} samples")

train_dataset = Dataset.from_list(train_data)
val_dataset = Dataset.from_list(val_data)

# Show emotions
print("\n Emotions:")
emotions = {}
for item in train_data:
    e = item.get('emotion', 'unknown')
    emotions[e] = emotions.get(e, 0) + 1
for e, c in sorted(emotions.items(), key=lambda x: x[1], reverse=True):
    print(f"   {e}: {c}")

# FIXED FORMATTING FOR LLAMA
def format_for_llama_tamil(example):
    """
    Format Tamil data for LLaMA (NOT Gemma!).
    This FIXES the 'user' output bug.
    """
    # Extract user input ROBUSTLY
    prompt = example.get('prompt', '')
    user_query = extract_user_input_robust(prompt)

    # Get response and emotion
    bot_response = example.get('target', '')
    emotion = example.get('emotion', 'unknown')

    # Add emotion conditioning
    if emotion != 'unknown':
        user_query = f"[உணர்வு: {emotion}] {user_query}"

    # LLaMA 3.2 format (SIMPLE AND WORKS!)
    text = (
        f"<|begin_of_text|><|start_header_id|>system<|end_header_id|>\n\n"
        f"You are an empathetic Tamil therapy assistant. Respond in Tamil with care and understanding.<|eot_id|>"
        f"<|start_header_id|>user<|end_header_id|>\n\n"
        f"{user_query}<|eot_id|>"
        f"<|start_header_id|>assistant<|end_header_id|>\n\n"
        f"{bot_response}<|eot_id|>"
    )

    return {
        "text": text,
        "emotion": emotion,
        "user_input": user_query,
        "target_response": bot_response
    }

# Format
print("\nFormatting for LLaMA...")
train_dataset = train_dataset.map(format_for_llama_tamil)
val_dataset = val_dataset.map(format_for_llama_tamil)

# Save for eval
train_dataset_eval = train_dataset
val_dataset_eval = val_dataset

# Keep only text
train_dataset = train_dataset.remove_columns(
    [c for c in train_dataset.column_names if c != "text"]
)
val_dataset = val_dataset.remove_columns(
    [c for c in val_dataset.column_names if c != "text"]
)

print("\n DATA READY")
print(f"   Train: {len(train_dataset)}")
print(f"   Val: {len(val_dataset)}")

# Show sample
print("\n Sample:")
print("="*70)
print(train_dataset_eval[0]['text'][:300] + "...")
print("="*70)

LOADING TAMIL DATA (ROBUST)

Loading training data...
    788 samples

Loading validation data...
    98 samples

 Emotions:
   neutral: 114
   angry: 114
   surprised: 114
   sad: 114
   happy: 114
   fearful: 113
   disgust: 105

Formatting for LLaMA...


Map:   0%|          | 0/788 [00:00<?, ? examples/s]

Map:   0%|          | 0/98 [00:00<?, ? examples/s]


 DATA READY
   Train: 788
   Val: 98

 Sample:
<|begin_of_text|><|start_header_id|>system<|end_header_id|>

You are an empathetic Tamil therapy assistant. Respond in Tamil with care and understanding.<|eot_id|><|start_header_id|>user<|end_header_id|>

[உணர்வு: neutral] நான் ஒரு கஃபேயில் அமர்ந்திருக்கிறேன்.<|eot_id|><|start_header_id|>assistant<|...


## CELL 6: Training Config (8 Epochs for Better Tamil)

In [ ]:
print("="*70)
print("TRAINING CONFIG (OPTIMIZED FOR TAMIL)")
print("="*70)

training_args = SFTConfig(
    output_dir=OUTPUT_DIR,

    # Dataset
    dataset_text_field="text",
    max_length=1024,
    packing=False,

    # Training (MORE EPOCHS for Tamil)
    num_train_epochs=8,        # ← INCREASED from 5
    per_device_train_batch_size=2,
    per_device_eval_batch_size=2,
    gradient_accumulation_steps=8,

    # Learning
    learning_rate=2e-4,
    lr_scheduler_type="cosine",
    warmup_ratio=0.1,

    # Optimization
    optim="paged_adamw_32bit",
    weight_decay=0.02,
    max_grad_norm=0.5,

    # Precision
    bf16=torch.cuda.is_bf16_supported(),
    fp16=not torch.cuda.is_bf16_supported(),

    # Memory
    gradient_checkpointing=True,

    # Eval & save
    eval_strategy="steps",
    eval_steps=25,
    save_strategy="steps",
    save_steps=25,
    save_total_limit=5,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",

    # Logging
    logging_steps=5,
    logging_first_step=True,
    report_to="none",
)

print("\n Summary:")
print(f"   Model: LLaMA 3.2-3B")
print(f"   Epochs: {training_args.num_train_epochs}")
print(f"   LoRA rank: 96 (Tamil-optimized)")

warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


TRAINING CONFIG (OPTIMIZED FOR TAMIL)

 Summary:
   Model: LLaMA 3.2-3B
   Epochs: 8
   LoRA rank: 96 (Tamil-optimized)


##  CELL 7: Start Training!

In [ ]:
print("="*70)
print("STARTING TAMIL THERAPY TRAINING")
print("="*70)

trainer = SFTTrainer(
    model=model,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    args=training_args,
    processing_class=tokenizer,
)

print("\n Training...")

trainer.train()

print("\n TRAINING DONE!")

STARTING TAMIL THERAPY TRAINING


Adding EOS to train dataset:   0%|          | 0/788 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/788 [00:00<?, ? examples/s]

Truncating train dataset:   0%|          | 0/788 [00:00<?, ? examples/s]

Adding EOS to eval dataset:   0%|          | 0/98 [00:00<?, ? examples/s]

...

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': 128009, 'pad_token_id': 128009}.



 Training...


Step,Training Loss,Validation Loss
25,0.505817,0.489481
50,0.388823,0.378988
75,0.311094,0.366352
100,0.295185,0.358148
125,0.219661,0.369064
150,0.238268,0.355415
175,0.149955,0.396074
200,0.151581,0.392476
225,0.092839,0.427014
250,0.088537,0.435613


/usr/local/lib/python3.12/dist-packages/peft/utils/other.py:1394: UserWarning: Unable to fetch remote file due to the following error 401 Client Error. (Request ID: Root=1-69abf718-43abe4d56ca21b854f063f98;a3e017b0-809f-41ea-b90c-704a027857ca)

Cannot access gated repo for url https://huggingface.co/meta-llama/Llama-3.2-3B-Instruct/resolve/main/config.json.
Access to model meta-llama/Llama-3.2-3B-Instruct is restricted. You must have access to it and be authenticated to access it. Please log in. - silently ignoring the lookup for the file config.json in meta-llama/Llama-3.2-3B-Instruct.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/peft/utils/save_and_load.py:295: UserWarning: Could not find a config file in meta-llama/Llama-3.2-3B-Instruct - will assume that the vocabulary was not modified.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/peft/utils/other.py:1394: UserWarning: Unable to fetch remote file due to the following error 401 Client Error. (Request ID: Root


 TRAINING DONE!


##  CELL 8: Save Model

In [ ]:
SAVE_DIR = f"{OUTPUT_DIR}/final_model"
print(f"Saving to: {SAVE_DIR}")
model.save_pretrained(SAVE_DIR)
tokenizer.save_pretrained(SAVE_DIR)
print(" Saved!")

try:
    DRIVE_DIR = "/content/drive/MyDrive/llama3-tamil-therapy2"
    model.save_pretrained(DRIVE_DIR)
    tokenizer.save_pretrained(DRIVE_DIR)
    print(f" Also saved to Drive: {DRIVE_DIR}")
except:
    print(" Drive save skipped")

Saving to: ./llama3-tamil-therapy-fixed/final_model


/usr/local/lib/python3.12/dist-packages/peft/utils/other.py:1394: UserWarning: Unable to fetch remote file due to the following error 401 Client Error. (Request ID: Root=1-69abfebd-4ac800353af638497b8c5334;5b004ff4-cf02-4cf9-9e7c-fdc666ceff62)

Cannot access gated repo for url https://huggingface.co/meta-llama/Llama-3.2-3B-Instruct/resolve/main/config.json.
Access to model meta-llama/Llama-3.2-3B-Instruct is restricted. You must have access to it and be authenticated to access it. Please log in. - silently ignoring the lookup for the file config.json in meta-llama/Llama-3.2-3B-Instruct.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/peft/utils/save_and_load.py:295: UserWarning: Could not find a config file in meta-llama/Llama-3.2-3B-Instruct - will assume that the vocabulary was not modified.
  warnings.warn(


 Saved!


/usr/local/lib/python3.12/dist-packages/peft/utils/other.py:1394: UserWarning: Unable to fetch remote file due to the following error 401 Client Error. (Request ID: Root=1-69abfec2-08c6a20748169cf374b5472d;1556c455-6c58-4723-83c9-d1c70773abe3)

Cannot access gated repo for url https://huggingface.co/meta-llama/Llama-3.2-3B-Instruct/resolve/main/config.json.
Access to model meta-llama/Llama-3.2-3B-Instruct is restricted. You must have access to it and be authenticated to access it. Please log in. - silently ignoring the lookup for the file config.json in meta-llama/Llama-3.2-3B-Instruct.
  warnings.warn(


 Also saved to Drive: /content/drive/MyDrive/llama3-tamil-therapy2


##  CELL 9: Test Generation (FIXED - No more 'user' output!)

In [ ]:
print("="*70)
print("TESTING TAMIL GENERATION (FIXED!)")
print("="*70)

def generate_tamil_response(user_input, emotion=None):
    """
    FIXED generation function.
    This will NOT output 'user' anymore!
    """
    # Add emotion if provided
    if emotion:
        user_input = f"[உணர்வு: {emotion}] {user_input}"

    # Build prompt
    prompt = (
        f"<|begin_of_text|><|start_header_id|>system<|end_header_id|>\n\n"
        f"You are an empathetic Tamil therapy assistant. Respond in Tamil with care and understanding.<|eot_id|>"
        f"<|start_header_id|>user<|end_header_id|>\n\n"
        f"{user_input}<|eot_id|>"
        f"<|start_header_id|>assistant<|end_header_id|>\n\n"
    )

    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=300,      # INCREASED
            min_new_tokens=40,       # FORCE minimum
            temperature=0.8,         # Slightly higher
            top_p=0.92,             # Adjusted
            top_k=50,
            do_sample=True,
            repetition_penalty=1.15, # Prevent loops
            length_penalty=1.0,      # Encourage longer
            no_repeat_ngram_size=3,
            pad_token_id=tokenizer.eos_token_id,
            eos_token_id=tokenizer.eos_token_id,
        )

    # Decode
    response = tokenizer.decode(outputs[0], skip_special_tokens=True)

    # ROBUST EXTRACTION (prevents 'user' bug)
    # Remove everything before assistant response
    if 'assistant' in response:
        parts = response.split('assistant')
        if len(parts) > 1:
            response = parts[-1].strip()

    # Remove any trailing tokens
    if '<|eot_id|>' in response:
        response = response.split('<|eot_id|>')[0].strip()

    # Remove system/user text if present
    if 'system' in response.lower():
        response = response.split('system', 1)[-1].strip()
    if 'user' in response.lower() and len(response) < 20:
        # If response is just 'user', generate again
        return "[Generation failed - retrying needed]"

    return response.strip()

# TEST CASES
print("\n Testing...\n")

tests = [
    ("எனக்கு மிகவும் வருத்தமாக உள்ளது", "வருத்தம்"),
    ("எனக்கு கவலையாக இருக்கிறது", "கவலை"),
    ("நான் தனிமையாக உணர்கிறேன்", "தனிமை"),
    ("எனக்கு கோபமாக இருக்கிறது", "கோபம்"),
]

for i, (text, emotion) in enumerate(tests, 1):
    print(f"Test {i}/{len(tests)}:")
    print(f"User: {text}")

    response = generate_tamil_response(text, emotion=emotion)

    print(f"Bot: {response}")
    print(f"Length: {len(response.split())} words")

    # Check quality
    has_tamil = any('\u0B80' <= c <= '\u0BFF' for c in response)
    is_complete = len(response) > 50

    print(f"Tamil: {'Yes' if has_tamil else 'No'}")
    print(f"Complete: {'Yes' if is_complete else 'No'}")
    print("-" * 70)
    print()

print(" Testing done!")

TESTING TAMIL GENERATION (FIXED!)

 Testing...

Test 1/4:
User: எனக்கு மிகவும் வருத்தமாக உள்ளது
Bot: அந்த வெறுப்பு உఙ்களுடன் இயல்பா಩தா? ஒரிஜினல் அழுவதை விட, சில நேரங்ஙளில் 'நா எಪ்சன்டஸ்ட்' எ-ஹூர் ஏகௌமோயிக்ளெ தீர முய லா-கி-ஆனுகுள் ஒ-நியோ புரோ-ஃபெ-ரி வு-எஸ்-ஷா -எ வி பிடுங் கொளலவோ!
Length: 23 words
Tamil: Yes
Complete: Yes
----------------------------------------------------------------------

Test 2/4:
User: எனக்கு கவலையாக இருக்கிறது
Bot: அந்த சிந்மய பிரச்சினை ஒரே ஆட்களை, இతர விஷயை-உন்னதமா, வரம் அல்லத்தொன்றை -தூண்டும்போதியுள்ளதੁப் பொ. 'அ' எ.னி எ..
Length: 14 words
Tamil: Yes
Complete: Yes
----------------------------------------------------------------------

Test 3/4:
User: நான் தனிமையாக உணர்கிறேன்
Bot: அல்லது சோகப்படுவதா? ஆழ்ந்த உళ்ளுறை ஸ்திரத்தரங்கள் உন்நதி சா கா. எந்ன ஒரு நேরத் கூட்டத്തிற்கு அடியில் ஐதீக்க எ.எஃப்.ஐ., அ.சி, ஹா.ஏ., பரிஷ்க்ரம சிறிய புதிய 1-800 ஜன்னலின்‌விதி..
Length: 22 words
Tamil: Yes
Complete: Yes
----------------------------------------------------------------------

##  CELL 10: Full Evaluation

In [ ]:
print("="*70)
print("FULL EVALUATION")
print("="*70)

import evaluate

# Generate on validation
print("\nGenerating predictions...")
predictions = []
references = []
emotions = []

for i, ex in enumerate(val_dataset_eval):
    user_input = ex['user_input']
    ref = ex['target_response']
    emotion = ex['emotion']

    pred = generate_tamil_response(user_input, emotion=emotion if emotion != 'unknown' else None)

    predictions.append(pred)
    references.append(ref)
    emotions.append(emotion)

    if (i + 1) % 10 == 0:
        print(f"   {i + 1}/{len(val_dataset_eval)}...")

print(f"\n {len(predictions)} predictions")

# ROUGE
rouge = evaluate.load('rouge')
rouge_results = rouge.compute(predictions=predictions, references=references)

print("\n ROUGE:")
print(f"   ROUGE-1: {rouge_results['rouge1']:.4f}")
print(f"   ROUGE-L: {rouge_results['rougeL']:.4f}")

# Quality
avg_len = np.mean([len(p.split()) for p in predictions])
has_tamil = [any('\u0B80' <= c <= '\u0BFF' for c in p) for p in predictions]
tamil_pct = (sum(has_tamil) / len(has_tamil)) * 100
complete = [len(p) > 50 for p in predictions]
complete_pct = (sum(complete) / len(complete)) * 100

print("\n Quality:")
print(f"   Avg length: {avg_len:.1f} words")
print(f"   Tamil: {tamil_pct:.1f}%")
print(f"   Complete: {complete_pct:.1f}%")

# Show examples
print("\n Examples:")
for i in range(min(5, len(predictions))):
    print(f"\n{i+1}. User: {val_dataset_eval[i]['user_input'][:50]}...")
    print(f"   Pred: {predictions[i][:100]}...")

print("\n" + "="*70)
print(" EVALUATION DONE!")
print("="*70)

FULL EVALUATION

Generating predictions...
   10/98...
   20/98...
   30/98...
   40/98...
   50/98...
   60/98...
   70/98...
   80/98...
   90/98...

 98 predictions



 ROUGE:
   ROUGE-1: 0.0000
   ROUGE-L: 0.0000

 Quality:
   Avg length: 18.6 words
   Tamil: 100.0%
   Complete: 99.0%

 Examples:

1. User: [உணர்வு: angry] சிறிய அசௌகரியங்களால் நான் எரிச்சலட...
   Pred: மிகை ஊடுருவுதலை ఆபத்தா சூழ்நில�ailை  உங் கட்டுপ்பாట்க்கு ஏற்ற வழி அ-வெறுமை.ஒரே சமநிறு வீர் அ உ திரு ...

2. User: [உணர்வு: disgust] வாக்குறுதிகளை எனக்கு நானே நிறைவே...
   Pred: அதை 'ஒருவர் உలகம் எಟுக്കவில்லை' எ঩்று சொல்வதాல் பிரதీக సரியா? ஏத் தகுతியை, உತவி அலெய்டரை (வுட்ஹா வௌச...

3. User: [உணர்வு: angry] அவர்கள் அனைவருக்கும் முன்னால் என்ன...
   Pred: அந்த இயற்கை-ஆபதி மாுஸ்டரின் பின்வாఙ்ஙுவோம்ஃ உங் கவிழ்ந்ந விரக্তியை 'அ'வ்வூரை, ஒரச் சீறியா அ... எ. கே...

4. User: [உணர்வு: angry] அவர்கள் எனது யோசனையை எடுத்து அது த...
   Pred: இதை 'எம்'பாக மறுசீரமைక்க ஒழிய நேரியபூரி. கொஞ்சம்தா கடினமா, அ-வே-முப்பொ எ-கஷியுமிங்?...

5. User: [உணர்வு: happy] நான் ஜிம்மில் ஒரு சிறந்த உடற்பயிற்...
   Pred: ஆழம் பரவுவது எளிதாక இரீடிங்-ஏக்ஸர். புதிய சூழ்நிலைகளை 'என்ன' எ அ அ, இ இ, எ 'ஐ', 'ஓ', அ 'ய'....

 EVALUAT